In [0]:
dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

environment = dbutils.widgets.get("environment").lower()

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "catalog": "salesjson_dev"
    },
    "prod": {
        "catalog": "salesjson_prod"
    }
}

env = config[environment]

catalog = env["catalog"]

# Bronze
bronze_table = f"{catalog}.bronze.orders_raw"

# Silver
silver_table = f"{catalog}.silver.orders"
rejected_table = f"{catalog}.silver.rejected_orders"

# Gold
daily_table = f"{catalog}.gold.daily_sales_summary"
category_table = f"{catalog}.gold.category_sales_summary"
customer_table = f"{catalog}.gold.customer_sales_summary"

print("=" * 60)
print("SALES JSON - METADATA DOCUMENTATION")
print("=" * 60)
print(f"Environment       : {environment}")
print(f"Catalog           : {catalog}")
print(f"Bronze table      : {bronze_table}")
print(f"Silver table      : {silver_table}")
print(f"Rejected table    : {rejected_table}")
print(f"Daily Gold        : {daily_table}")
print(f"Category Gold     : {category_table}")
print(f"Customer Gold     : {customer_table}")
print("=" * 60)

In [0]:
spark.sql(f"""
COMMENT ON TABLE {bronze_table}
IS 'Raw JSON sales orders incrementally ingested from Azure Data Lake Storage using Databricks Auto Loader and managed file events. The table preserves source data and adds technical ingestion metadata for traceability and reprocessing.'
""")


bronze_column_comments = {
    "order_id":
        "Unique identifier of the sales order.",

    "customer_id":
        "Identifier of the customer who placed the order.",

    "order_timestamp":
        "Timestamp provided by the source system when the order was created.",

    "country":
        "Country associated with the sales order.",

    "city":
        "City associated with the sales order.",

    "product_id":
        "Unique identifier of the ordered product.",

    "product_name":
        "Name of the ordered product.",

    "category":
        "Business category assigned to the product.",

    "quantity":
        "Quantity of product units included in the order.",

    "unit_price":
        "Unit price provided by the source system.",

    "discount":
        "Discount percentage provided by the source system.",

    "payment_method":
        "Payment method used for the order.",

    "status":
        "Current status of the sales order.",

    "shipping_priority":
        "Shipping priority provided by the source system after schema evolution.",

    "_rescued_data":
        "Fields that Auto Loader could not reconcile with the inferred schema.",

    "source_file":
        "Name of the JSON source file from which the record was ingested.",

    "source_file_path":
        "Full Azure Data Lake Storage path of the source JSON file.",

    "source_file_modification_time":
        "Last modification timestamp of the source file in Azure Data Lake Storage.",

    "ingestion_timestamp":
        "Timestamp associated with the Bronze ingestion execution."
}

existing_columns = {
    field.name
    for field in spark.table(bronze_table).schema.fields
}

for column_name, comment in bronze_column_comments.items():
    if column_name in existing_columns:
        spark.sql(f"""
            ALTER TABLE {bronze_table}
            ALTER COLUMN `{column_name}`
            COMMENT '{comment}'
        """)

In [0]:


spark.sql(f"""
COMMENT ON TABLE {silver_table}
IS 'Validated, cleaned, standardized, deduplicated, and enriched sales orders derived from the Bronze JSON ingestion layer. This table contains records that passed all defined business data-quality rules and is designed for downstream analytics and Gold transformations.'
""")

spark.sql(f"""
COMMENT ON TABLE {rejected_table}
IS 'Sales orders rejected during Silver processing because they failed one or more business data-quality rules. Rejected records are retained for traceability, auditing, and troubleshooting.'
""")

silver_column_comments = {
    "gross_amount":
        "Gross order amount calculated before applying discounts.",

    "discount_amount":
        "Total monetary discount applied to the order.",

    "net_amount":
        "Net order amount after applying discounts.",

    "silver_processing_timestamp":
        "Timestamp associated with the Silver transformation execution.",

    "shipping_priority":
        "Normalized shipping priority propagated from the source after schema evolution."
}

existing_columns = {
    field.name
    for field in spark.table(silver_table).schema.fields
}

for column_name, comment in silver_column_comments.items():
    if column_name in existing_columns:
        spark.sql(f"""
            ALTER TABLE {silver_table}
            ALTER COLUMN `{column_name}`
            COMMENT '{comment}'
        """)

        

spark.sql(f"""
ALTER TABLE {rejected_table}
ALTER COLUMN `rejection_reason`
COMMENT 'Business data-quality rule that caused the record to be rejected from the validated Silver dataset.'
""")



In [0]:
spark.sql(f"""
COMMENT ON TABLE {daily_table}
IS 'Daily sales performance summary containing order volumes, customer counts, units sold, gross revenue, discounts, net revenue, and order-value statistics. Designed for time-series analysis, Databricks Genie, dashboards, and BI reporting.'
""")

spark.sql(f"""
COMMENT ON TABLE {category_table}
IS 'Sales performance aggregated by product category, including order volumes, customers, units sold, revenue, discounts, and order-value statistics. Designed for product-category analysis and Databricks Genie.'
""")

spark.sql(f"""
COMMENT ON TABLE {customer_table}
IS 'Customer sales summary aggregated by customer and country. Contains purchasing activity, revenue, order-value metrics, and customer activity timestamps. Designed for customer analytics, Databricks Genie, and governed data-access scenarios.'
""")


daily_column_comments = {
    "sale_date":
        "Calendar date used for daily sales aggregation.",

    "total_orders":
        "Number of distinct sales orders for the day.",

    "total_customers":
        "Number of distinct customers who placed orders during the day.",

    "total_units":
        "Total quantity of product units sold during the day.",

    "gross_revenue":
        "Total revenue before discounts.",

    "total_discount":
        "Total monetary discount applied to sales.",

    "net_revenue":
        "Total revenue after discounts.",

    "average_order_value":
        "Average net amount per sales order.",

    "min_order_value":
        "Lowest net order amount for the day.",

    "max_order_value":
        "Highest net order amount for the day.",

    "gold_processing_timestamp":
        "Timestamp associated with the Gold aggregation execution."
}

for column_name, comment in daily_column_comments.items():
    spark.sql(f"""
        ALTER TABLE {daily_table}
        ALTER COLUMN `{column_name}`
        COMMENT '{comment}'
    """)

category_column_comments = {
    "category":
        "Product category used for sales aggregation.",

    "total_orders":
        "Number of distinct sales orders containing products in the category.",

    "total_customers":
        "Number of distinct customers purchasing from the category.",

    "total_units":
        "Total quantity of product units sold in the category.",

    "gross_revenue":
        "Total category revenue before discounts.",

    "total_discount":
        "Total monetary discount applied to category sales.",

    "net_revenue":
        "Total category revenue after discounts.",

    "average_order_value":
        "Average net order amount for the category.",

    "min_order_value":
        "Lowest net order amount associated with the category.",

    "max_order_value":
        "Highest net order amount associated with the category.",

    "gold_processing_timestamp":
        "Timestamp associated with the Gold aggregation execution."
}

for column_name, comment in category_column_comments.items():
    spark.sql(f"""
        ALTER TABLE {category_table}
        ALTER COLUMN `{column_name}`
        COMMENT '{comment}'
    """)

    # COMMAND ----------

customer_column_comments = {
    "customer_id":
        "Unique identifier of the customer.",

    "country":
        "Country associated with the customer sales activity.",

    "total_orders":
        "Number of distinct orders placed by the customer.",

    "total_units":
        "Total quantity of product units purchased by the customer.",

    "gross_revenue":
        "Total customer sales value before discounts.",

    "total_discount":
        "Total monetary discount applied to the customer sales.",

    "total_spent":
        "Total net amount spent by the customer after discounts.",

    "average_order_value":
        "Average net order amount for the customer.",

    "first_order_timestamp":
        "Timestamp of the earliest order included in the customer summary.",

    "last_order_timestamp":
        "Timestamp of the most recent order included in the customer summary.",

    "gold_processing_timestamp":
        "Timestamp associated with the Gold aggregation execution."
}

for column_name, comment in customer_column_comments.items():
    spark.sql(f"""
        ALTER TABLE {customer_table}
        ALTER COLUMN `{column_name}`
        COMMENT '{comment}'
    """)